# Wikipedia Search Engine with LangChain + FAISS + Groq

Pipeline:
1. Load Wikipedia articles for a topic
2. Split into chunks & embed with `sentence-transformers/all-MiniLM-L6-v2`
3. Store in a local FAISS vector index
4. Answer questions with Groq's Llama-3.1-8b via an LCEL retrieval chain

**Fixes applied vs original:**
- Removed pinned old versions — uses current LangChain 1.x
- Replaced broken `wikipedia` package with `wikipedia-api`
- Replaced deprecated `WikipediaLoader` with a direct `wikipedia-api` loader
- Replaced removed `langchain.chains.RetrievalQA` with modern LCEL pipe syntax
- Replaced deprecated `HuggingFaceEmbeddings` from community with `langchain-huggingface`
- Added `--only-binary=:all:` to avoid C-compiler errors on Windows/Python 3.13

In [ ]:
# ============================================================
# STEP 0 — Install packages
# Run once, then restart the kernel before continuing.
# ============================================================
%pip install -q --only-binary=:all: \
    langchain \
    langchain-core \
    langchain-community \
    langchain-text-splitters \
    langchain-groq \
    langchain-huggingface \
    pydantic \
    faiss-cpu \
    sentence-transformers \
    transformers \
    huggingface_hub \
    wikipedia-api \
    python-dotenv

In [ ]:
# ============================================================
# STEP 0b — Verify installs (run after kernel restart)
# ============================================================
import langchain, langchain_core, pydantic, faiss, wikipediaapi
print(f"langchain:      {langchain.__version__}")
print(f"langchain-core: {langchain_core.__version__}")
print(f"pydantic:       {pydantic.__version__}")
print(f"faiss:          OK")
print(f"wikipedia-api:  OK")

In [ ]:
# ============================================================
# Imports
# ============================================================
import os
import warnings
warnings.filterwarnings("ignore")

import wikipediaapi                                          # replaces broken WikipediaLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings     # replaces langchain_community.embeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser   # replaces RetrievalQA chain

print("All imports OK")

## Set Groq API Key

Get a free key at https://console.groq.com, then set it one of these ways:

- **Windows CMD:** `set GROQ_API_KEY=your_key_here` then relaunch VS Code from that terminal  
- **Mac/Linux:** `export GROQ_API_KEY=your_key_here`  
- **`.env` file** in the same folder as this notebook:  
  ```
  GROQ_API_KEY=your_key_here
  ```

In [ ]:
# Load from .env file if present
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # dotenv not installed, rely on environment variable

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not set. See instructions above.")

print("API key loaded.")

## Step 1 — Load Wikipedia Articles

Uses `wikipedia-api` directly instead of the broken `WikipediaLoader`.

In [ ]:
def load_wikipedia(topic: str, lang: str = "en", max_sections: int = 5) -> list[Document]:
    """
    Fetch a Wikipedia page and return it as a list of LangChain Documents.
    One Document per section, plus one for the page summary.
    """
    wiki = wikipediaapi.Wikipedia(
        language=lang,
        user_agent="RAG-Pipeline/1.0"
    )
    page = wiki.page(topic)

    if not page.exists():
        raise ValueError(f"Wikipedia page not found for: '{topic}'")

    docs = []

    # Page summary as first document
    if page.summary.strip():
        docs.append(Document(
            page_content=page.summary,
            metadata={"title": page.title, "section": "summary", "source": page.fullurl}
        ))

    # Each section as a separate document
    for section in list(page.sections)[:max_sections]:
        if section.text.strip():
            docs.append(Document(
                page_content=section.text,
                metadata={"title": page.title, "section": section.title, "source": page.fullurl}
            ))

    return docs


query_topic = input("Enter topic to search on Wikipedia: ")
documents = load_wikipedia(query_topic)

print(f"Loaded {len(documents)} section(s) from Wikipedia")
print(documents[0].page_content[:500])

## Step 2 — Split into Chunks

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)
print(f"Total chunks: {len(chunks)}")

## Step 3 — Create Embeddings (Local, No API Needed)

In [ ]:
# First download may take ~1 minute (model is ~90MB)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("Embeddings model loaded")

## Step 4 — Build FAISS Vector Store

In [ ]:
vectorstore = FAISS.from_documents(chunks, embeddings)
print(f"FAISS index built with {vectorstore.index.ntotal} vectors")

## Step 5 — Create Retriever

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)
print("Retriever ready")

## Step 6 — Load Groq LLM

In [ ]:
llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model_name="llama-3.1-8b-instant",
    temperature=0
)
print("Groq LLM loaded")

## Step 7 — Build LCEL Retrieval Chain

`RetrievalQA` was removed in LangChain 1.x.  
The modern replacement is the **LCEL pipe (`|`) syntax**.

In [ ]:
# Prompt template
prompt = ChatPromptTemplate.from_template("""
Answer the question using ONLY the context provided below.
If the answer is not in the context, say "I don't have enough information to answer that."

Context:
{context}

Question: {question}
""")

# Helper: join retrieved docs into one string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# LCEL chain:  question --> retrieve --> format --> prompt --> LLM --> parse
qa_chain = (
    {
        "context":  retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("QA chain ready!")

## Step 8 — Ask Questions

In [ ]:
def ask(question: str) -> str:
    """Run a question through the QA chain and return the answer."""
    return qa_chain.invoke(question)


# Interactive loop — type 'exit' to quit
while True:
    query = input("Ask a question (or 'exit' to quit): ").strip()
    if query.lower() == "exit":
        break
    print("\nAnswer:", ask(query), "\n")